# Clean Nepali Movies Data


In [1]:
from pathlib import Path
import json
import re
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "data cleaning.ipynb").exists():
    NOTEBOOK_DIR = Path("/Users/prabinthakur/code101/test_apps/nepali movies review/nlp projects/text preprocessing ").resolve()

RAW_PATH = NOTEBOOK_DIR.parent / "data acquisition" / "movies_raw.jsonl"
OUTPUT_DIR = NOTEBOOK_DIR


In [2]:
records = []
malformed = []

with RAW_PATH.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, 1):
        if not line.strip():
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError as error:
            malformed.append((line_number, str(error)))
            continue
        if record.get("status", "ok") == "ok":
            records.append(record)

seen = set()
clean_records = []
for record in records:
    movie_id = record.get("id")
    slug = record.get("slug")
    key = ("id", str(movie_id).strip()) if movie_id not in (None, "") else ("slug", str(slug).strip() if slug else "")
    if key not in seen:
        seen.add(key)
        clean_records.append(record)

print(f"Parsed/status-ok: {len(records)}")
print(f"Malformed JSON lines: {len(malformed)}")
for line_number, error in malformed:
    print(f"  line {line_number}: {error}")
print(f"After deduplication: {len(clean_records)}")


Parsed/status-ok: 1803
Malformed JSON lines: 0
After deduplication: 1803


In [3]:
def text(value):
    return value.strip() if isinstance(value, str) else None


def string_list(value):
    if not isinstance(value, list):
        value = [value] if isinstance(value, str) else []
    return list(dict.fromkeys(item.strip() for item in value
                             if isinstance(item, str) and item.strip()))


def people(value):
    if isinstance(value, dict):
        value = [value]
    if not isinstance(value, list):
        return []
    return list(dict.fromkeys(text(person.get("name")) for person in value
                             if isinstance(person, dict) and text(person.get("name"))))


def nullable_int(value):
    if value is None or isinstance(value, bool):
        return pd.NA
    number = pd.to_numeric(value, errors="coerce")
    return pd.NA if pd.isna(number) else int(number)


def duration(value):
    if value is None or isinstance(value, bool):
        return pd.NA
    if isinstance(value, (int, float)):
        return pd.NA if pd.isna(value) else int(round(float(value)))

    value = str(value).strip()
    number = pd.to_numeric(value, errors="coerce")
    if not pd.isna(number):
        return int(round(float(number)))

    match = re.fullmatch(r"P(?:(\d+(?:\.\d+)?)D)?(?:T(?:(\d+(?:\.\d+)?)H)?(?:(\d+(?:\.\d+)?)M)?(?:(\d+(?:\.\d+)?)S)?)?", value, re.I)
    if not match:
        return pd.NA
    days, hours, minutes, seconds = (float(part) if part else 0 for part in match.groups())
    return int(round(days * 1440 + hours * 60 + minutes + seconds / 60))


def score(value):
    match = re.search(r"(-?\d+(?:\.\d+)?)", str(value)) if value is not None else None
    return float(match.group(1)) if match else pd.NA


movie_rows = []
review_rows = []

for record in clean_records:
    movie_id_value = record.get("id")
    slug = text(record.get("slug"))
    movie_id = str(movie_id_value).strip() if movie_id_value not in (None, "") else slug

    date_published = pd.to_datetime(record.get("datePublished"), errors="coerce", format="mixed")
    release_year = nullable_int(record.get("catalog_year"))
    if pd.isna(release_year) and not pd.isna(date_published):
        release_year = int(date_published.year)

    movie_rows.append({
        "movie_id": movie_id,
        "slug": slug,
        "title": text(record.get("title")) or text(record.get("ld_name")),
        "alternate_names": string_list(record.get("alternateName")),
        "description": text(record.get("description")),
        "genres": string_list(record.get("genre")),
        "date_published": date_published,
        "release_year": release_year,
        "duration_mins": duration(record.get("duration")),
        "directors": people(record.get("director")),
        "cast": people(record.get("actor")),
        "total_ratings": nullable_int(record.get("catalog_total_ratings")),
        "reviews_count": nullable_int(record.get("reviews_count")),
    })

    reviews = record.get("reviews")
    if not isinstance(reviews, list):
        reviews = []
    for review in reviews:
        if not isinstance(review, dict):
            continue
        review_text = text(review.get("text"))
        review_rows.append({
            "movie_id": movie_id,
            "slug": slug,
            "author": text(review.get("author")),
            "review_date": pd.to_datetime(review.get("date"), errors="coerce", format="mixed"),
            "score_10": score(review.get("score")),
            "text": review_text,
            "has_written_review": bool(review_text and review_text != "User shared a rating only."),
        })

movies_df = pd.DataFrame(movie_rows)
reviews_df = pd.DataFrame(review_rows)

movies_df["date_published"] = pd.to_datetime(movies_df["date_published"], errors="coerce")
for column in ("release_year", "duration_mins", "total_ratings", "reviews_count"):
    movies_df[column] = pd.to_numeric(movies_df[column], errors="coerce").astype("Int64")
reviews_df["review_date"] = pd.to_datetime(reviews_df["review_date"], errors="coerce")
reviews_df["score_10"] = pd.to_numeric(reviews_df["score_10"], errors="coerce")
reviews_df["has_written_review"] = reviews_df["has_written_review"].astype("bool")

print("movies_df:", movies_df.shape)
print("reviews_df:", reviews_df.shape)


movies_df: (1803, 13)
reviews_df: (389, 7)


In [4]:
print("=== Data Quality Audit ===")
print("movies_df shape:", movies_df.shape)
print("reviews_df shape:", reviews_df.shape)

for name, frame in (("movies_df", movies_df), ("reviews_df", reviews_df)):
    def empty_count(column):
        total = 0
        for value in column:
            if isinstance(value, (list, tuple)):
                total += len(value) == 0
            elif isinstance(value, str):
                total += value == ""
            else:
                total += bool(pd.isna(value))
        return total

    counts = pd.DataFrame({
        "null_count": frame.isna().sum(),
        "empty_count": frame.apply(empty_count),
    })
    print(f"\n{name} null/empty counts:")
    print(counts)

print("\nGenre frequencies:")
print(movies_df["genres"].explode().dropna().value_counts().to_string())

boilerplate = movies_df["description"].fillna("").str.contains("is a movie directed by", case=False, regex=False)
print("\nBoilerplate descriptions:", int(boilerplate.sum()))

review_counts = reviews_df["has_written_review"].value_counts().rename(index={True: "written", False: "rating-only"})
print("\nReview types:")
print(review_counts.to_string())

movies_csv = movies_df.copy()
for column in ("alternate_names", "genres", "directors", "cast"):
    movies_csv[column] = movies_csv[column].apply(lambda values: "   ".join(values))

movies_df.to_parquet(OUTPUT_DIR / "movies_clean.parquet", index=False)
reviews_df.to_parquet(OUTPUT_DIR / "reviews_clean.parquet", index=False)
movies_csv.to_csv(OUTPUT_DIR / "movies_clean.csv", index=False)

print("\nExported:")
print(OUTPUT_DIR / "movies_clean.parquet")
print(OUTPUT_DIR / "reviews_clean.parquet")
print(OUTPUT_DIR / "movies_clean.csv")


=== Data Quality Audit ===
movies_df shape: (1803, 13)
reviews_df shape: (389, 7)

movies_df null/empty counts:
                 null_count  empty_count
movie_id                  0            0
slug                      0            0
title                     0            0
alternate_names           0           86
description               0         1481
genres                    0            0
date_published            0            0
release_year              0            0
duration_mins            93           93
directors                 0           61
cast                      0            1
total_ratings             0            0
reviews_count             0            0

reviews_df null/empty counts:
                    null_count  empty_count
movie_id                     0            0
slug                         0            0
author                       0            0
review_date                  0            0
score_10                    20           20
text               